Decoding the parameters that control and change the models output

In [4]:
from openai import OpenAI
import numpy as np

client = OpenAI(base_url = "http://123.176.46.139:5173/v1", api_key="e")
MODEL = "Qwen/Qwen3-VL-32B-Instruct"

In [5]:
prompt = "Write one short sentence about the ocean."

for T in [ 0.0 , 0.3 , 0.7, 1.2, 2.0]:
    outs = []
    for _ in range(10):
        r = client.chat.completions.create(
            model = MODEL,
            messages = [{"role" : "user", "content" : prompt}],
            temperature = T, max_tokens=30
        )
        outs.append(r.choices[0].message.content.strip())
    print(f"\n === T = {T} === unique: {len(set(outs))} /10")
    for o in outs:
        print(f"  -: {o}")


 === T = 0.0 === unique: 1 /10
  -: The ocean stretches vast and deep, cradling life beneath its shimmering surface.
  -: The ocean stretches vast and deep, cradling life beneath its shimmering surface.
  -: The ocean stretches vast and deep, cradling life beneath its shimmering surface.
  -: The ocean stretches vast and deep, cradling life beneath its shimmering surface.
  -: The ocean stretches vast and deep, cradling life beneath its shimmering surface.
  -: The ocean stretches vast and deep, cradling life beneath its shimmering surface.
  -: The ocean stretches vast and deep, cradling life beneath its shimmering surface.
  -: The ocean stretches vast and deep, cradling life beneath its shimmering surface.
  -: The ocean stretches vast and deep, cradling life beneath its shimmering surface.
  -: The ocean stretches vast and deep, cradling life beneath its shimmering surface.

 === T = 0.3 === unique: 2 /10
  -: The ocean stretches endlessly, its deep blue waves whispering secrets o

In [6]:
for p in [ 0.1, 0.5, 0.9 , 1.0]:
    outs = [
        client.chat.completions.create(
            model = MODEL, messages=[{"role" : "user", "content": prompt}],
            temperature = 1.0, top_p = p , max_tokens=30
        ).choices[0].message.content.strip() for _ in range(10)
    ]
    print(f"top_p = {p} unique: {len(set(outs))} / 10")


top_p = 0.1 unique: 1 / 10
top_p = 0.5 unique: 1 / 10
top_p = 0.9 unique: 9 / 10
top_p = 1.0 unique: 9 / 10


In [7]:
for k in [ 1 , 5 , 20 , 100]:
    outs = [
        client.chat.completions.create(
            model = MODEL, messages = [ {"role" : "user", "content" : prompt}],
            temperature = 1.0, extra_body = {"top_k" : k}, max_tokens = 30
        ).choices[0].message.content.strip() for _ in range(10)
    ]
    print(f"top k={k} , unique : {len(set(outs))} / 10")


top k=1 , unique : 1 / 10
top k=5 , unique : 7 / 10
top k=20 , unique : 7 / 10
top k=100 , unique : 5 / 10


In [8]:
r = client.chat.completions.create(
    model = MODEL,
    messages = [ { "role" : "user", "content" : " My favorite season is "}],
    temperature = 1.0, max_tokens = 1, 
    logprobs = True, top_logprobs = 10,
)

top = r.choices[0].logprobs.content[0].top_logprobs
for t in top: 
    print(f"{t.token!r:12} logprob = {t.logprob:7.3f} prob={np.exp(t.logprob):.4f}")

'My'         logprob =  -0.001 prob=0.9992
'That'       logprob =  -7.751 prob=0.0004
'Sure'       logprob =  -8.376 prob=0.0002
'Your'       logprob =  -9.376 prob=0.0001
'It'         logprob =  -9.876 prob=0.0001
'You'        logprob = -11.626 prob=0.0000
'Spring'     logprob = -11.626 prob=0.0000
'(My'        logprob = -11.751 prob=0.0000
'"My'        logprob = -11.876 prob=0.0000
'I'          logprob = -13.501 prob=0.0000


In [9]:
tokens = [t.token for t in top]
logits = np.array([t.logprob for t in top])   # logprobs as proxy logits — valid, softmax is shift-invariant

def softmax(x):
    e = np.exp(x - x.max()); return e / e.sum()

for T in [0.3, 0.7, 1.0, 1.5, 2.0, 30]:
    probs = softmax(logits / T)
    print(f"\nT={T}")
    for tok, p in sorted(zip(tokens, probs), key=lambda x: -x[1]):
        print(f"  {tok!r:12} {p:.3f} {'█' * int(p * 40)}")


T=0.3
  'My'         1.000 ███████████████████████████████████████
  'That'       0.000 
  'Sure'       0.000 
  'Your'       0.000 
  'It'         0.000 
  'You'        0.000 
  'Spring'     0.000 
  '(My'        0.000 
  '"My'        0.000 
  'I'          0.000 

T=0.7
  'My'         1.000 ███████████████████████████████████████
  'That'       0.000 
  'Sure'       0.000 
  'Your'       0.000 
  'It'         0.000 
  'You'        0.000 
  'Spring'     0.000 
  '(My'        0.000 
  '"My'        0.000 
  'I'          0.000 

T=1.0
  'My'         0.999 ███████████████████████████████████████
  'That'       0.000 
  'Sure'       0.000 
  'Your'       0.000 
  'It'         0.000 
  'You'        0.000 
  'Spring'     0.000 
  '(My'        0.000 
  '"My'        0.000 
  'I'          0.000 

T=1.5
  'My'         0.986 ███████████████████████████████████████
  'That'       0.006 
  'Sure'       0.004 
  'Your'       0.002 
  'It'         0.001 
  'You'        0.000 
  'Spring'     0.000 
  

In [10]:
r = client.completions.create(
    model=MODEL,
    prompt="The weather today is",
    temperature=1.0, max_tokens=1, logprobs=10,
)
top = {k: v for k, v in r.choices[0].logprobs.top_logprobs[0].items()}
tokens = list(top.keys())
logits = np.array(list(top.values()))


def softmax(x):
    e = np.exp(x - x.max()); return e / e.sum()

for T in [0.3, 0.7, 1.0, 1.5, 2.0, 30]:
    probs = softmax(logits / T)
    print(f"\nT={T}")
    for tok, p in sorted(zip(tokens, probs), key=lambda x: -x[1]):
        print(f"  {tok!r:12} {p:.3f} {'█' * int(p * 40)}")


T=0.3
  ' very'      0.776 ███████████████████████████████
  ' sunny'     0.146 █████
  ' clear'     0.018 
  ' quite'     0.018 
  ' ____'      0.012 
  ' not'       0.012 
  ' '          0.005 
  ' hot'       0.005 
  ' cold'      0.003 
  ' ______'    0.003 

T=0.7
  ' very'      0.376 ███████████████
  ' sunny'     0.184 ███████
  ' clear'     0.075 ███
  ' quite'     0.075 ███
  ' ____'      0.063 ██
  ' not'       0.063 ██
  ' '          0.044 █
  ' hot'       0.044 █
  ' cold'      0.037 █
  ' ______'    0.037 █

T=1.0
  ' very'      0.273 ██████████
  ' sunny'     0.165 ██████
  ' clear'     0.088 ███
  ' quite'     0.088 ███
  ' ____'      0.078 ███
  ' not'       0.078 ███
  ' '          0.061 ██
  ' hot'       0.061 ██
  ' cold'      0.054 ██
  ' ______'    0.054 ██

T=1.5
  ' very'      0.202 ████████
  ' sunny'     0.145 █████
  ' clear'     0.096 ███
  ' quite'     0.096 ███
  ' ____'      0.088 ███
  ' not'       0.088 ███
  ' '          0.074 ██
  ' hot'       0.074 ██

The Paramerters : Temperature, top k and top p. They are used to filter the final vocab for the next token prediction. Temperature acts a divisor that divides the logits before they ran through the softmax layer and get converted to probabilities. Higher T narrows the difference or gap between logits, making the tokens more diluted ? I am not sure of wording, I understood the underlaying thing, when we divide the logits with high temperature the change is more to the logit with higher score and does littel to nothing to the logits will littel score , in a way giving chance for the tokens that are scored initially low, where as low t , increase that difference or gap even more between token to token, making the token with higher score even higher value.

Top K : this filter runs after softmax converted the logits into predicitons, actively selecting the k predictions from the top. 

Top P : A value between 0 and 1 , runs after k in vllm, it further filters the top k tokens seleced with their aggregate is within the or just crossed the p . if p is 0.9 and top 3 tokens aggregate already crossed that value it will discard in a way makes the remaining tokens probabiltiies to 0.

Decoding parameters: temperature, top-k, top-p. They control how the next token is chosen from the vocabulary distribution. Important: the final step is NOT "take the highest probability" — it's a weighted random sample (a dice roll) among the surviving tokens. These three knobs shape that roll.

Temperature acts as a divisor applied to the logits before they go through softmax. Every logit is divided by the same T. Higher T shrinks the gaps between logits, so after softmax the distribution flattens — the top token's lead is compressed and low-scoring tokens get real probability mass, making output more diverse/random. Lower T widens the gaps, sharpening the distribution so the top token dominates and output becomes more deterministic. T=0 is special-cased to pure argmax (greedy). The key correction to my instinct: temperature doesn't "penalize" the top logit — it divides all of them evenly; a big gap divided by T just shrinks more in absolute terms than a small one, and softmax turns that smaller gap into a smaller probability lead.

top-k and top-p are filters that decide which tokens are ELIGIBLE for the dice roll (applied after temperature). top-k = keep the k highest tokens, ban the rest — a fixed cutoff by count. top-p (nucleus) = keep tokens in descending order until their cumulative probability reaches p, ban the rest — an adaptive cutoff by mass (p is between 0 and 1). top-k always keeps exactly k regardless of the model's certainty; top-p flexes — 1–2 tokens when the model is confident (peaked), many when it's unsure (flat). That adaptiveness is why top-p is usually preferred. vLLM applies them in order: temperature → top-k → top-p → weighted sample.

Structured-output takeaway: at T=0.7 there's still mass on non-optimal tokens, so at each JSON-structural position there's a small chance of sampling a format-breaking token; over a long output those risks compound into occasional malformed JSON. Fix: T=0 (optionally top_k=1) for anything structured — it collapses the dice roll back to argmax (deterministic) — and save temperature for prose.

`p=0` is a degenerate edge case: the cumulative-mass rule says "keep tokens until you reach 0% mass" — but the very first token already exceeds 0, so you keep **just that one token**. That collapses to greedy/argmax, same as `top_k=1` or `T=0`.

In practice most implementations (vLLM included) guard against literal 0 and treat very small `p` as "keep only the top token," so `p=0` and `p→0` both give you deterministic single-token output.